In [ ]:
from pyspark.sql import SparkSession
import hail as hl
import dxpy

builder = (
    SparkSession
    .builder
    .enableHiveSupport()
)
spark = builder.getOrCreate()
hl.init(sc=spark.sparkContext)

In [ ]:
tb = hl.import_table(f'file:///mnt/project/DNM/trios/out/trios_diffs_vep.vcf', 
                     delimiter = ' ', missing = '.', no_header = True,
                     types = {'f1': hl.tint32})

tb.describe()

In [ ]:
tb = tb.annotate(locus = hl.locus('chr'+tb.f0, tb.f1, reference_genome = 'GRCh38'))
tb = tb.annotate(alleles = [tb.f2, tb.f3])
tb = tb.key_by('locus', 'alleles')

tb.describe()

In [ ]:
tb.show()

In [ ]:
ann_tb = hl.vep(tb, "file:///mnt/project/vep.json")

In [ ]:
ann_tb.show()

In [ ]:
ann_tb.export('trios_diffs_vep.tsv.bgz')

In [ ]:
!hdfs dfs -get trios_diffs_vep.tsv.bgz .